In [1]:
# NOTEBOOK 26 - cross-city generalisation: Delhi-trained WA-TGCN v3 on Kolkata and Mumbai
# Zero-shot  : the saved Delhi checkpoints (dynamic / static / no-graph) and the Delhi scalers, applied unchanged.
# Few-shot   : each Delhi checkpoint fine-tuned for a few epochs on the city's own Sep-Nov 2025 data.
# Test       : 8 Dec 2025 - 9 Apr 2026 (the same calendar period as the Delhi test set), scored on hours that were really
#              observed (imputed PM2.5 hours are excluded from the scores).
# Needs: notebook 24 (city data) and notebook 25 (static / no-graph checkpoints).
# HOW TO CHECK: this cell only imports and sets options.

In [2]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV     = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL  = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
ERA5_WIND_NPY   = os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')
DYN_ADJ_OM_NPY  = os.path.join(V2_DIR, 'dynamic_adj_om.npy')
STATIC_ADJ_NPY  = os.path.join(V2_DIR, 'static_adj_matrix.npy')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
LOSS_VOL_ALPHA = 1.0
SEEDS = [42, 7, 123, 2024, 999]

cfg = json.load(open(os.path.join(MODEL_DIR, 'v3_config.json')))
FEATURE_COLS = cfg['feature_cols']; F = cfg['in_features']
HIDDEN_DIM, GRU_LAYERS = cfg['hidden_dim'], cfg['gru_layers']
ATTN_HEADS, DROPOUT, GCN_HOPS = cfg['attn_heads'], cfg['dropout'], cfg['gcn_hops']
N_FUT = cfg['n_fut']
print(f'Device {DEVICE} | F={F} | n_fut={N_FUT} | seeds {SEEDS}')

Device cuda | F=26 | n_fut=10 | seeds [42, 7, 123, 2024, 999]


In [3]:
class WATGCNv3Ablate(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon,
                 n_stations, dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        self.register_buffer('static_adj', sa, persistent=False)

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(h))) + h
        else:
            if self.graph_mode == 'dynamic':
                A = self._n(At + I.view(1, 1, n, n))
            else:
                A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        else:
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        fut_emb = self.fut_enc(Fut)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, fut_emb], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

In [4]:
def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs(); quad = torch.clamp(absr, max=delta)
    per = (0.5*quad**2 + delta*(absr - quad)).mean(dim=(1, 2))
    return (per * w).mean() / w.mean().clamp(min=1e-6)

def run_epoch(model, loader, opt=None, sched=None):
    tr = opt is not None; model.train(tr); tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched: sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, A = [], []
    for X, At, Afc, Fut, anc, y, w in loader:
        P.append(model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), anc.to(DEVICE)).cpu().numpy())
        A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)

def sc(pl, al):
    pr, ar = from_log(pl), from_log(al)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())))

def train_one(mode, seed, max_epochs=60, patience=12, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    model = build(mode)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = len(tr_ld) * max_epochs; warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s/warm if s < warm else 0.5*(1 + np.cos(np.pi*(s-warm)/max(1, steps-warm))))
    best, best_state, ctr = 1e9, None, 0
    for ep in range(1, max_epochs+1):
        t0 = time.time()
        trl = run_epoch(model, tr_ld, opt, sched); vl = run_epoch(model, va_ld)
        if vl < best - 1e-5: best, best_state, ctr = vl, copy.deepcopy(model.state_dict()), 0
        else: ctr += 1
        if ep % 4 == 0 or ctr >= patience:
            print(f'    ep {ep:>2} train {trl:.4f} val {vl:.4f} ({time.time()-t0:.0f}s) best {best:.4f}')
        if ctr >= patience: break
    model.load_state_dict(best_state)
    return model
print('ready')

ready


In [5]:
# scalers, city data and window datasets
import pickle
feat_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v3_feature_scaler.pkl'), 'rb'))
log_scaler  = pickle.load(open(os.path.join(SCALER_DIR, 'v3_log_scaler.pkl'), 'rb'))
def from_log(a): return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):   return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)

WORK = os.path.join(V2_DIR, 'crosscity_cache')
CITIES = ['Kolkata', 'Mumbai', 'Ahmedabad', 'Patna']
RERUN = []        # put a city name here to recompute it even if it already has saved results
TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
FUT_IDX = [FEATURE_COLS.index(c) for c in TIME_COLS + ['u_wind', 'v_wind', 'wspd']]
assert len(FUT_IDX) == N_FUT

class City:
    def __init__(self, name):
        z = np.load(os.path.join(WORK, f'city_{name}.npz'))
        self.name = name
        self.X = z['X_raw']; self.era5 = z['era5']; self.dyn = z['dyn_adj']; self.static = z['static_adj']; self.obs = z['obs_mask']
        self.names = list(z['names']); self.grid = pd.DatetimeIndex(z['grid'])
        self.T, self.n, _ = self.X.shape
        self.Xs = feat_scaler.transform(self.X.reshape(-1, F)).reshape(self.T, self.n, F).astype(np.float32)
        self.Yl = to_log(self.X[:, :, 0])
        idx = lambda s: int(np.searchsorted(self.grid, pd.Timestamp(s)))
        W = INPUT_WINDOW + HORIZON
        t_test, t_val = idx('2025-12-08'), idx('2025-11-20')
        self.test_starts  = np.arange(t_test, self.T - W + 1)
        self.val_starts   = np.arange(t_val, t_test - W + 1)          # forecast ends before 8 Dec: no overlap with test
        self.train_starts = np.arange(0, t_val - W + 1)
class CityDS(Dataset):
    def __init__(self, c, starts, weights=None):
        self.c, self.s = c, np.asarray(starts, np.int64)
        self.w = (weights if weights is not None else np.ones(len(self.s), np.float32)).astype(np.float32)
    def __len__(self): return len(self.s)
    def __getitem__(self, k):
        c = self.c; g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(c.Xs[g:i1].copy()), torch.from_numpy(c.dyn[g:i1].copy()), torch.from_numpy(c.dyn[i1:i2].copy()),
                torch.from_numpy(np.ascontiguousarray(c.Xs[i1:i2][:, :, FUT_IDX])), torch.from_numpy(c.Yl[i1-1].copy()),
                torch.from_numpy(c.Yl[i1:i2].copy()), torch.tensor(self.w[k], dtype=torch.float32))

cities = {n: City(n) for n in CITIES}
for n, c in cities.items():
    print(f'{n}: {c.n} stations, {c.T} hours | windows train {len(c.train_starts)} / val {len(c.val_starts)} / test {len(c.test_starts)}'
          f' | PM2.5 observed {c.obs.mean():.2f}')

# HOW TO CHECK: Kolkata 10 stations, Mumbai 27 stations; train about 1,900 windows, val about 400, test about 2,900.

Kolkata: 10 stations, 5305 hours | windows train 1849 / val 361 / test 2882 | PM2.5 observed 0.89
Mumbai: 21 stations, 5305 hours | windows train 1849 / val 361 / test 2882 | PM2.5 observed 0.87
Ahmedabad: 9 stations, 5305 hours | windows train 1849 / val 361 / test 2882 | PM2.5 observed 0.88
Patna: 6 stations, 5305 hours | windows train 1849 / val 361 / test 2882 | PM2.5 observed 0.86


C:\Users\jlakk_j7eh5fd\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# model building and checkpoints
def build_city(mode, c):
    return WATGCNv3Ablate(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, c.n, DROPOUT, N_FUT, GCN_HOPS,
                          graph_mode=mode, static_adj=c.static).to(DEVICE)
def ckpt(mode, sd):
    name = {'dynamic': f'watgcn_v3_seed{sd}.pt', 'static': f'watgcn_v3static_seed{sd}.pt', 'none': f'watgcn_v3none_seed{sd}.pt'}[mode]
    f = os.path.join(MODEL_DIR, name)
    assert os.path.exists(f), f'missing {name} - run notebook 25 first'
    return f

def metrics(pred_log, act_log, mask=None):
    p, a = from_log(pred_log), from_log(act_log)
    if mask is not None:
        p, a = p[mask], a[mask]
    p, a = p.ravel(), a.ravel()
    return dict(rmse=float(np.sqrt(np.mean((p - a) ** 2))), mae=float(np.mean(np.abs(p - a))),
                r2=float(1 - np.sum((p - a) ** 2) / np.sum((a - a.mean()) ** 2)), bias=float(np.mean(p - a)))

def target_mask(c, starts):
    return np.stack([c.obs[g + INPUT_WINDOW: g + INPUT_WINDOW + HORIZON] for g in starts])

def horizon_rmse(pred_log, act_log, mask):
    p, a = from_log(pred_log), from_log(act_log)
    return [float(np.sqrt(np.mean(((p[:, h] - a[:, h])[mask[:, h]]) ** 2))) for h in (0, 5, 11, 17, 23)]

def finetune(mode, sd, c, max_epochs=15, patience=4, lr=3e-4):
    torch.manual_seed(sd); np.random.seed(sd)
    m = build_city(mode, c); m.load_state_dict(torch.load(ckpt(mode, sd), map_location=DEVICE))
    sw = np.array([(c.X[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON, :, 0].max(0) - c.X[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON, :, 0].min(0)).mean()
                   for g in c.train_starts], dtype=np.float32)
    w = np.clip(1.0 + LOSS_VOL_ALPHA * (sw / (np.median(sw) + 1e-6)), 0.5, 4.0).astype(np.float32); w *= len(w) / w.sum()
    tr = DataLoader(CityDS(c, c.train_starts, w), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    va = DataLoader(CityDS(c, c.val_starts), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    best, state, ctr = run_epoch(m, va), copy.deepcopy(m.state_dict()), 0       # epoch 0 = the zero-shot model itself
    for ep in range(1, max_epochs + 1):
        run_epoch(m, tr, opt); vl = run_epoch(m, va)
        if vl < best - 1e-5: best, state, ctr = vl, copy.deepcopy(m.state_dict()), 0
        else: ctr += 1
        if ctr >= patience: break
    m.load_state_dict(state)
    return m
print('ready')

ready


In [7]:
# run everything: persistence, zero-shot and few-shot for the three Delhi-trained graph variants, 5 seeds each
MODES = ['none', 'static', 'dynamic']
_rf = os.path.join(RESULTS_DIR, 'crosscity_results.json')
results = json.load(open(_rf)) if os.path.exists(_rf) else {}   # keeps earlier cities, only new ones are computed
for cname, c in cities.items():
    if cname in results and cname not in RERUN:
        print(f'{cname}: using saved results'); continue
    te_ld = DataLoader(CityDS(c, c.test_starts), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    mask = target_mask(c, c.test_starts)
    anc = np.stack([c.Yl[g + INPUT_WINDOW - 1] for g in c.test_starts])
    results[cname] = {}
    preds = {}
    preds['persistence'] = np.repeat(anc[:, None, :], HORIZON, axis=1)
    for mode in MODES:
        ps = []
        for sd in SEEDS:
            m = build_city(mode, c); m.load_state_dict(torch.load(ckpt(mode, sd), map_location=DEVICE))
            p, act = predict(m, te_ld); ps.append(p); del m
        preds[f'zero-shot {mode}'] = np.mean(ps, axis=0)
        fs = []
        for sd in SEEDS:
            m = finetune(mode, sd, c); p, act = predict(m, te_ld); fs.append(p); del m; torch.cuda.empty_cache()
        preds[f'few-shot {mode}'] = np.mean(fs, axis=0)
        print(f'{cname}: {mode} done')
    for k, p in preds.items():
        r = metrics(p, act, mask); r['by_horizon'] = horizon_rmse(p, act, mask)
        results[cname][k] = r

for cname in CITIES:
    print(f'\n=== {cname}  ({cities[cname].n} stations; scores on observed hours only; persistence is the bar to beat) ===')
    print(f"{'method':22}{'RMSE':>8}{'MAE':>8}{'R2':>8}{'bias':>8}    +1h   +6h  +12h  +18h  +24h")
    for k, r in results[cname].items():
        print(f"{k:22}{r['rmse']:>8.2f}{r['mae']:>8.2f}{r['r2']:>8.3f}{r['bias']:>8.1f}  " + ' '.join(f'{x:5.1f}' for x in r['by_horizon']))
    for stage in ('zero-shot', 'few-shot'):
        gap = results[cname][f'{stage} static']['rmse'] - results[cname][f'{stage} dynamic']['rmse']
        gap0 = results[cname][f'{stage} none']['rmse'] - results[cname][f'{stage} dynamic']['rmse']
        print(f'  {stage}: dynamic vs static {gap:+.2f} | dynamic vs no-graph {gap0:+.2f} ug/m3 (positive = dynamic is better)')

json.dump(results, open(os.path.join(RESULTS_DIR, 'crosscity_results.json'), 'w'), indent=2)
print('\nsaved results/crosscity_results.json')

# HOW TO READ: the bar to beat is persistence. A large negative bias in zero-shot means the Delhi scale does not transfer
# (expected for Mumbai, whose PM2.5 is lower); few-shot should remove most of it. Differences between dynamic, static and
# no-graph below about 1 ug/m3 are inside seed noise.

Kolkata: using saved results
Mumbai: using saved results
Ahmedabad: none done
Ahmedabad: static done
Ahmedabad: dynamic done
Patna: none done
Patna: static done
Patna: dynamic done

=== Kolkata  (10 stations; scores on observed hours only; persistence is the bar to beat) ===
method                    RMSE     MAE      R2    bias    +1h   +6h  +12h  +18h  +24h
persistence              44.60   24.44   0.211     1.1   21.5  45.3  50.9  47.9  36.8
zero-shot none           30.46   18.31   0.632     4.3   24.0  29.7  31.5  31.7  31.3
few-shot none            29.97   16.89   0.644    -0.8   21.8  29.2  31.3  31.5  30.6
zero-shot static         31.22   18.31   0.613     0.1   23.2  30.3  32.5  32.8  32.3
few-shot static          30.92   17.05   0.621    -4.7   22.8  30.1  32.1  32.6  31.8
zero-shot dynamic        31.35   18.49   0.610    -0.1   23.2  30.4  32.8  33.0  32.2
few-shot dynamic         29.95   16.73   0.644    -1.7   21.8  29.1  31.1  31.6  30.9
  zero-shot: dynamic vs static -0.12

In [8]:
# figure: RMSE per city for persistence, zero-shot and few-shot
fig, axes = plt.subplots(1, len(CITIES), figsize=(5.2 * len(CITIES), 3.8), sharey=False)
for ax, cname in zip(np.atleast_1d(axes), CITIES):
    labels = ['persistence', 'zero-shot none', 'zero-shot static', 'zero-shot dynamic', 'few-shot none', 'few-shot static', 'few-shot dynamic']
    vals = [results[cname][k]['rmse'] for k in labels]
    cols = ['gray'] + ['tab:blue'] * 3 + ['tab:green'] * 3
    ax.bar(range(len(labels)), vals, color=cols)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels([l.replace('-shot ', '\n') for l in labels], fontsize=8)
    ax.set_title(f'{cname} ({cities[cname].n} stations)'); ax.set_ylabel('test RMSE (ug/m3)')
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, 'crosscity_rmse.png'), dpi=160); plt.show()
print('saved plots/crosscity_rmse.png')

saved plots/crosscity_rmse.png


C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_14044\1713723158.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, 'crosscity_rmse.png'), dpi=160); plt.show()
